# Simulate bending and twisting a sheet to form a Moebius strip

In [ ]:
import sys; sys.path.extend(['..', '../validations/'])
import MeshFEM
import mesh, elastic_sheet, tri_mesh_viewer, sim_utils, sheet_convergence
import py_newton_optimizer
import numpy as np

In [ ]:
L = 12
es = sheet_convergence.getSheet(0.05, maxArea=1e-2, L=L, useNeoHookean=True)
es.setDeformedPositions(es.getRestPositions() - np.mean(es.mesh().bbox, axis=0))

In [ ]:
# Determine the variables at the left and right ends.
leftEdgePosVars  = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_X, displacementsOnly=True)
rightEdgePosVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_X, displacementsOnly=True)
leftEdgeVars     =  sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_X)
rightEdgeVars    =  sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_X)

In [ ]:
# Set up equilibrium solver with Dirichlet conditions only.
eq = es.equilibriumOptimizer(fixedVars=leftEdgeVars + rightEdgeVars)
eq.options.verbose = 0

In [ ]:
def bendFrame(t):
    def bend(X, subtendedAngle):
        x, y, z = X[:, 0], X[:, 1], X[:, 2]
        x_mid = 0.5 * (x.min() + x.max())
        y_mid = 0.5 * (y.min() + y.max())
        R = L / subtendedAngle
        thetaForX = lambda x: -(subtendedAngle / L) * (x - x_mid)
        theta = thetaForX(x)
        ptForTheta = lambda t, y: (-R * np.sin(t), y, R * (1 - np.cos(t)))
        # Determine the translation placing the sheet midpoint
        # at its final `subtendedAngle = 2pi` location.
        theta_mid = np.array(ptForTheta(thetaForX(x_mid), y_mid))
        theta_mid_final = np.array([-L / (2 * np.pi) * np.sin(-np.pi), y_mid, -L / (2 * np.pi)])
        return np.column_stack(ptForTheta(theta, y)) + theta_mid_final - theta_mid

    es.setDeformedPositions(bend(es.getRestPositions() - np.mean(es.mesh().bbox, axis=0), (1 - t) * 0.00001 + t * 2*np.pi))
    es.initializeMidedgeNormals()
    eq.optimize()

def twistFrame(t, x_cylinder):
    import scipy
    R = scipy.spatial.transform.Rotation.from_rotvec([t * np.pi / 2, 0, 0]).as_matrix()
    x = es.getVars()
    P = x_cylinder[rightEdgePosVars].reshape(-1, 3)
    c = [0, 0, L / (2 * np.pi)]
    x[rightEdgePosVars] = ((P - c) @ R.T + c).ravel()
    P = x_cylinder[leftEdgePosVars].reshape(-1, 3)
    x[leftEdgePosVars] = ((P - c) @ R + c).ravel()
    es.setVars(x)
    eq.optimize()
    
def smoothstep(x): return x*x*(3.0-2.0*x)

In [ ]:
bendFrame(0)
v = tri_mesh_viewer.Viewer(es)
v.show()

In [ ]:
for t in np.linspace(0, 1, 60):
    bendFrame(smoothstep(t))
    v.update()

In [ ]:
bendFrame(1)
x_cylinder = es.getVars()

In [ ]:
for t in np.linspace(0, 1, 60):
    twistFrame(smoothstep(t), x_cylinder)
    v.update()

In [ ]:
for t in np.linspace(1, 0, 60):
    twistFrame(smoothstep(t), x_cylinder)
    v.update()